In [38]:
from sls_client import get_sls_data_by_query
from datetime import datetime

major_query = """
inboundFlag:major_price_msgid_ and DELETE | select 
cast(json_extract_scalar(body,'$.old[0].admin_id') as bigint) admin_id 
,json_extract_scalar(body,'$.old[0].sku') sku 
,json_extract_scalar(body,'$.old[0].pd_name') pd_name
,json_extract_scalar(body,'$.old[0].area_no') area_no 
,json_extract_scalar(body,'$.old[0].valid_time') valid_time
,json_extract_scalar(body,'$.old[0].invalid_time') invalid_time
,json_extract_scalar(body,'$.old[0].cost') "成本"
,json_extract_scalar(body,'$.old[0].interest_rate') "报价毛利率%"
,json_extract_scalar(body,'$.old[0].price') "最终价格"
,case json_extract_scalar(body,'$.old[0].direct') when '1' then '账期' when '2' then '现结' end "现结还是账期"
,case json_extract_scalar(body,'$.old[0].price_type') when '0' then '商城价' when '1' then '合同价（指定价）' when '2' then '合同价（毛利率）' when '3' then '商城价上浮' when '4' then '商城价下浮' when '5' then '商城价加价' when '6' then '商城价减价' end "报价方式"
from(select split(msg,'body:')[2] body,time from log) 
having admin_id in (216,477,1529,1712,1786,1819,2231,600810,1011888,1076311)  and invalid_time>'2025-03-25 00:00:00.0' and valid_time<'2025-03-21 00:00:00.0'
order by 1 limit 1000000
"""

deleted_major_price = get_sls_data_by_query(
    project="k8s-log-c7d28cba17d0a416ca4f52459592b8d38",
    logstore="prod-dts-stdout-log",
    query=major_query,
    from_time=datetime(2025, 3, 24, 21, 50, 0),
    to_time=datetime(2025, 3, 24, 22, 50, 0),
)

deleted_major_price.head(1)
# delete columns: __source__ and __time__
deleted_major_price = deleted_major_price.drop(columns=['__source__', '__time__'])
deleted_major_price.to_csv('deleted_major_price.csv')

即将获取数据: =====> 2025-03-24 21:50:00 2025-03-24 22:50:00 prod-dts-stdout-log: 
inboundFlag:major_price_msgid_ and DELETE | select 
cast(json_extract_scalar(body,'$.old[0].admin_i
>=====数条数:1238


In [39]:
admin_brief=deleted_major_price.groupby('admin_id').count().reset_index()
admin_brief['admin_id']=admin_brief['admin_id'].astype(int)
admin_brief

,admin_id,sku,pd_name,area_no,valid_time,invalid_time,成本,报价毛利率%,最终价格,现结还是账期,报价方式
0,1076311,141,141,141,141,141,141,141,141,141,141
1,1529,1,1,1,1,1,1,1,1,1,1
2,1712,64,64,64,64,64,64,64,64,64,64
3,1786,329,329,329,329,329,329,329,329,329,329
4,216,146,146,146,146,146,146,146,146,146,146
5,2231,62,62,62,62,62,62,62,62,62,62
6,477,9,9,9,9,9,9,9,9,9,9
7,600810,486,486,486,486,486,486,486,486,486,486


In [43]:
from odps_client import get_odps_sql_result_as_df

sql="""select a.*,b.large_area_name from summerfarm_tech.ods_area_df a 
left join summerfarm_tech.ods_large_area_df b on b.ds=max_pt('summerfarm_tech.ods_large_area_df') 
and a.large_area_no = b.large_area_no
where a.ds=max_pt('summerfarm_tech.ods_area_df');"""

area_df=get_odps_sql_result_as_df(sql)

2025-03-26 00:35:08 - INFO - Tunnel session created: <InstanceDownloadSession id=202503260035089bde321a0604b4b8 project_name=summerfarm_ds instance_id=20250325163459470go329uu7q6k>
2025-03-26 00:35:08 - INFO - sql:
select a.*,b.large_area_name from summerfarm_tech.ods_area_df a 
left join summerfarm_tech.ods_large_area_df b on b.ds=max_pt('summerfarm_tech.ods_large_area_df') 
and a.large_area_no = b.large_area_no
where a.ds=max_pt('summerfarm_tech.ods_area_df');
columns:Index(['id', 'area_no', 'area_name', 'admin_id', 'parent_no',
       'delivery_frequent', 'status', 'delivery_fee', 'info', 'address',
       'express_fee', 'delivery_rule', 'member_rule', 'company_account_id',
       'poi_note', 'type', 'free_day', 'mail_to_address', 'map_section',
       'origin_area_no', 'next_delivery_date', 'pay_channel', 'change_flag',
       'change_store_no', 'change_status', 'administrative_area',
       'create_time', 'support_add_order', 'update_support_add_order',
       'large_area_no', 'gr

In [41]:
admin_sql="select * from summerfarm_tech.ods_admin_df where ds=max_pt('summerfarm_tech.ods_admin_df');"

admin_df=get_odps_sql_result_as_df(admin_sql)
admin_df.head(1)

2025-03-26 00:32:46 - INFO - Tunnel session created: <InstanceDownloadSession id=20250326003246105ac90b06e41c97 project_name=summerfarm_ds instance_id=20250325163242528g1xbdb440gg>
2025-03-26 00:32:47 - INFO - sql:
select * from summerfarm_tech.ods_admin_df where ds=max_pt('summerfarm_tech.ods_admin_df');
columns:Index(['admin_id', 'create_time', 'login_fail_times', 'is_disabled',
       'username', 'password', 'login_time', 'realname', 'gender',
       'department', 'phone', 'kp', 'saler_id', 'saler_name', 'contract',
       'contract_method', 'name_remakes', 'operate_id', 'major_cycle',
       'close_order_type', 'cooperation_stage', 'update_time',
       'close_order_time', 'update_close_order_time', 'low_price_remainder',
       'not_included_area', 'sku_sorting', 'admin_type', 'admin_chain',
       'admin_grade', 'admin_switch', 'credit_code',
       'business_license_address', 'bill_to_pay', 'base_user_id', 'ds'],
      dtype='object')


,admin_id,create_time,login_fail_times,is_disabled,username,password,login_time,realname,gender,department,...,sku_sorting,admin_type,admin_chain,admin_grade,admin_switch,credit_code,business_license_address,bill_to_pay,base_user_id,ds
0,790582,2023-11-30 18:11:07,0,0,pei.li@lelechasc.com,fcea920f7412b5da7be0cf42b8c93759,NaT,杭州鲜沐科技有限公司,NaN,None,...,1,0.0,1.0,2.0,1,91330106MA27XXN62W,None,0,202705,20250324


In [46]:
area_df["area_no"] = area_df["area_no"].astype(int)
deleted_major_price["area_no"] = deleted_major_price["area_no"].astype(int)
admin_df["admin_id"] = admin_df["admin_id"].astype(int)
deleted_major_price["admin_id"] = deleted_major_price["admin_id"].astype(int)

merged_df = deleted_major_price.merge(
    area_df[["area_no", "area_name", "large_area_name"]], on="area_no", how="left"
)
merged_df = merged_df.merge(
    admin_df[["admin_id", "realname", "name_remakes"]], on="admin_id", how="left"
)
merged_df = merged_df.rename(
    columns={
        "admin_id": "大客户ID",
        "pd_name": "商品名字",
        "area_no": "运营服务区编号",
        "valid_time": "生效时间",
        "invalid_time": "失效时间",
        "realname": "大客户工商名",
        "name_remakes": "品牌",
        "area_name": "运营服务区名称",
        "large_area_name": "大区名称",
    }
)

for brand in merged_df["品牌"].unique():
    _df = merged_df[merged_df["品牌"] == brand]
    _df = _df.sort_values(by=["现结还是账期", "大区名称", "sku"])
    _df.to_csv(f"3.24删除的品牌报价单_{brand}.csv")

In [ ]:
from pandasql import sqldf

static_df = sqldf(
    "select admin_id,sku,count(1) as count from deleted_major_price group by admin_id,sku order by admin_id,sku"
)
static_df

In [ ]:
import pandas as pd
admin_info=pd.read_csv("./有多种合作方式的大客户列表.csv")
admin_info['admin_id']=admin_info['admin_id'].astype(int)
merge_df=admin_info.merge(admin_brief, on='admin_id', how='left')
merge_df

In [ ]:
from sls_client import get_sls_data_by_query

query="""
en:wbv and ap:/merchant/store/account/authorization/query-status | select case when status=0 then '已预录入但未授权' when status=1 then '取消授权' when status=2 then '已授权' else '未授权' end as "授权状态"
,date_of_year as "日期",uid,phone from (select cast(json_extract(json_extract_scalar(ai, '$.rt'),'$.data.status') as bigint) status, 
json_extract(json_extract_scalar(ai, '$.rt'),'$.data.displayed') displayed,uid,phone,date_format(__time__, '%Y%m%d') date_of_year from log limit 1000000) group by 1,2,3,4 limit 1000000
"""

from datetime import datetime,timedelta
yesterday = (datetime.now() - timedelta(days=1))
# 将yesterday的时分秒设置为0点0分0秒
yesterday = yesterday.replace(hour=0, minute=0, second=0, microsecond=0)
today=datetime.now()
# 将today的时分秒设置为0点0分0秒
today = today.replace(hour=0, minute=0, second=0, microsecond=0)

users_df=get_sls_data_by_query(project="xianmu-front-end-log",logstore="xm-mall", query=query, from_time=yesterday, to_time=today)

In [ ]:
pay_channel_query = """
ap:"/payment/union/pay" and en:mp| select case json_extract_scalar(ai, '$.rt.data.channelType') 
    when 'FIRE_FACE_WECHAT' then 'B2B支付' 
    when 'WECHAT_NATIVE' then '微信直连' 
    when 'DIN_WECHAT_MP' then '智付' 
    when 'BILL' then '账期' else '鲜沐卡' end as "支付渠道",uid,phone 
    from log where json_extract(ai, '$.rt.data') is not null 
    group by 1,2,3 limit 1000000
"""

pay_channel_df = get_sls_data_by_query(
    project="xianmu-front-end-log",
    logstore="xm-mall",
    query=pay_channel_query,
    from_time=yesterday,
    to_time=today,
)

In [ ]:
# 找出来哪些授权了，但是没有使用B2B支付的用户：


merged_df=pay_channel_df.merge(users_df, on="phone", how="inner")

merged_df[(merged_df['授权状态']=='已授权') & (merged_df['支付渠道']!='B2B支付')]

In [8]:
import pandas as pd

accounts=pd.read_csv("/Users/pengtang/Desktop/has_orders_last30days_account_id.csv")

In [ ]:
import requests

headers = {"token": "mall__eb3ee8f1-5668-4097-9656-d6b4e2a32c99"}

failed_accounts = []

import threading
import queue
from concurrent.futures import ThreadPoolExecutor

# 创建一个队列来存储结果
result_queue = queue.Queue()


def process_account(account_id, headers):
    """处理单个account的预录入"""
    url = f"https://h5.summerfarm.net/merchant/store/account/authorization/pre-entry?accountId={account_id}"
    try:
        result = requests.post(url=url, headers=headers).json()
        if not result["data"]:
            print(f"account预录入失败了: {account_id}, result: {result}")
            result_queue.put(account_id)  # 将失败的account_id放入队列
        else:
            print(f"account预录入成功了: {account_id}")
    except Exception as e:
        print(f"account预录入出错: {account_id}, error: {e}")
        result_queue.put(account_id)


# 使用ThreadPoolExecutor管理线程
with ThreadPoolExecutor(max_workers=3) as executor:
    for _index, row in accounts.iloc[
        2000:
    ].iterrows():  # 从第2001行开始，使用iloc[2000:]
        account_id = row["account_id"]
        executor.submit(process_account, account_id, headers)

# 从队列中获取所有失败的account_id
failed_accounts = []
while not result_queue.empty():
    failed_accounts.append(result_queue.get())

print(f"失败了{len(failed_accounts)}个account，失败的列表:{failed_accounts}")

In [ ]:
import pandas as pd

user_un_autherised_df = pd.read_csv("/Users/pengtang/Desktop/未授权的客户分析.csv")

user_un_autherised_df.describe()

In [ ]:
user_un_autherised_df.head(1)

In [9]:
city_df = (
    user_un_autherised_df.groupby(["门店城市"])
    .aggregate({"m_id": "count", "三十天订单数": "sum", "三十天下单GMV": "sum"})
    .reset_index()
)

city_df = city_df.sort_values(by=["三十天下单GMV"], ascending=False)

In [ ]:
total_gmv=user_un_autherised_df['三十天下单GMV'].sum()

city_df['GMV占比']=city_df['三十天下单GMV']/total_gmv
city_df

In [ ]:
for gmv in range(1000, 20000, 1000):
    _df = user_un_autherised_df[user_un_autherised_df["三十天下单GMV"] > gmv]
    print(
        f"最近30天GMV大于:{gmv}的客户数:{_df.shape[0]}, GMV占比:{round(100.0*_df['三十天下单GMV'].sum()/total_gmv,2)}%, 近30天GMV:{round(_df['三十天下单GMV'].sum()/10000)}万"
    )

In [ ]:
city_over_2000_df = (
    user_un_autherised_df[user_un_autherised_df["三十天下单GMV"] > 3000]
    .groupby(["门店城市"])
    .aggregate({"m_id": "count", "三十天订单数": "sum", "三十天下单GMV": "sum"})
    .reset_index()
)

city_over_2000_df = city_over_2000_df.sort_values(by=["三十天下单GMV"], ascending=False)
city_over_2000_df.columns = ["门店城市", "门店数", "三十天订单总数", "三十天总下单GMV"]
city_over_2000_df

In [ ]:
import json


# 从文件加载 JSON 数据: /Users/pengtang/major_price.json
with open("/Users/pengtang/major_price.json", "r") as f:
    data = json.load(f)

print(len(data['data']['list'][0]))

In [ ]:
import pandas as pd

major_price_df=pd.DataFrame(data['data']['list'][0])
major_price_df.describe()

In [ ]:
major_price_df.head(1)